# 05 - Data and model analysis

- **Part A: the data.** Solubility per solvent, overlaps (Venn), solvents that behave alike, effect of RT and molecular weight, composition of the training set.
- **Part B: the model.** Analyses that go beyond the overall score of notebook 04: per-solvent scores, confusion matrices, learning curve, calibration, top-k, feature importance, error analysis and effect of each plate.

Re-run after each new plate, once `scripts/data_prep/add_new_molecules.py` has been run. Part B takes a few minutes.

All figures share one style (`src/plot_style.py`), sized to go straight into slides: EPFL red for what we look at (plates, Model E), grey for references.

## Part A: the data

This part describes the data used to train the model: solubility in each solvent, overlaps between solvents, solvents that behave alike, the effect of RT and molecular weight, and the composition of the training set (original data vs plates).

It reads `data/interim/molecules_unique_updated.xlsx`, built by `scripts/data_prep/add_new_molecules.py`. Figures are saved to `results/figures/eda/data_analysis/`.

In [ ]:
import sys
from pathlib import Path
# Find the project root (the folder that contains src/config.py) and make src/ importable
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts" / "data_prep"))
from config import MOLECULES_UNIQUE_UPDATED, EXPERIMENTS, FIG_EDA, ensure_dir
from add_new_molecules import read_experiment, experiment_files
import plot_style as ps

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib_venn import venn3
from rdkit import Chem
from rdkit.Chem import Descriptors

ps.apply()
FIG_DATA = ensure_dir(FIG_EDA / "data_analysis")
SOLVENTS = ps.SOLVENTS
LABEL = ps.SOLVENT_LABELS
SOURCE_COLORS = {"original": ps.GREY, "plates": ps.RED}
REDS = LinearSegmentedColormap.from_list("epfl_reds", ["#FFFFFF", "#FFB3B3", "#FF0000", "#8C0000"])

def save(fig, name):
    fig.savefig(FIG_DATA / name)
    plt.show()

### 1. The dataset

In [ ]:
df = pd.read_excel(MOLECULES_UNIQUE_UPDATED)
df.columns = df.columns.str.strip()
df['CAS'] = df['CAS'].astype(str).str.strip()
df = df[df['solute_smiles'].notna() & df[SOLVENTS].notna().any(axis=1)].reset_index(drop=True)
df['MolWt'] = df['solute_smiles'].map(lambda s: Descriptors.MolWt(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else np.nan)

# Where does each molecule come from, and who gave its labels?
plate_records = []
for path in experiment_files():
    recs, _ = read_experiment(path)
    plate_records += [dict(r, plate=path.stem.replace('_output', '')) for r in recs]
plates = pd.DataFrame(plate_records)
plate_of = plates.groupby('cas')['plate'].first() if len(plates) else pd.Series(dtype=str)
df['source'] = np.where(df['CAS'].isin(plate_of.index), 'plates', 'original')
df['plate'] = df['CAS'].map(plate_of).fillna('original data')
df['hplc_column'] = df['Matrix ID Name'].str.replace('matrix ID ', '', regex=False)
n_src = df['source'].value_counts()

print(f"{len(df)} usable molecules")
display(pd.crosstab(df['plate'], df['hplc_column'], margins=True, margins_name='total'))
if len(plates):
    print(f"Plate labels: {plates['from_observation'].sum()} from your observation, "
          f"{(~plates['from_observation']).sum()} from the vision model prediction")

### 2. Probability of being soluble in each solvent

Share of soluble molecules for each solvent, separately for the original data and for the plates.

In [ ]:
stats = pd.DataFrame({
    'tested': df[SOLVENTS].notna().sum(),
    'soluble': (df[SOLVENTS] == 1).sum(),
    'P(soluble) all': df[SOLVENTS].mean(),
    'P(soluble) original': df[df['source'] == 'original'][SOLVENTS].mean(),
    'P(soluble) plates': df[df['source'] == 'plates'][SOLVENTS].mean(),
}).T
display(stats.style.format(lambda v: f"{v:.0%}" if v <= 1 else f"{v:.0f}", na_rep="-"))

fig, ax = plt.subplots(figsize=ps.HALF)
x = np.arange(len(SOLVENTS)); w = 0.38
ax.set_ylim(0, 1.12)
for i, src in enumerate(['original', 'plates']):
    vals = df[df['source'] == src][SOLVENTS].mean().values
    bars = ax.bar(x + (i - 0.5) * w, vals, w - 0.03, color=SOURCE_COLORS[src],
                  label=f"{src} ({n_src.get(src, 0)} molecules)")
    ps.label_bars(ax, bars, size=11)
ax.set_xticks(x, [LABEL[s] for s in SOLVENTS]); ax.yaxis.set_major_formatter(ps.pct)
ax.set_yticks([0, .25, .5, .75, 1]); ax.set_ylabel("soluble molecules")
ax.legend(loc='upper left', ncols=2, bbox_to_anchor=(0, 1.08))
save(fig, "p_soluble_per_solvent.png")

#### Histogram: in how many solvents is a molecule soluble?

In [ ]:
n_sol = (df[SOLVENTS] == 1).sum(axis=1)
counts = pd.crosstab(n_sol, df['source']).reindex(range(6), fill_value=0)
fig, ax = plt.subplots(figsize=ps.HALF)
bottom = np.zeros(6)
for src in ['original', 'plates']:
    if src in counts:
        ax.bar(counts.index, counts[src], 0.68, bottom=bottom, color=SOURCE_COLORS[src], label=src,
               edgecolor='white', linewidth=2)
        bottom += counts[src].values
for k, tot in enumerate(bottom):
    ax.text(k, tot + 1, f"{int(tot)}", ha='center', fontsize=12)
ax.set_xticks(range(6)); ax.set_xlabel("number of solvents the molecule dissolves in"); ax.set_ylabel("molecules")
ax.set_ylim(0, bottom.max() * 1.12); ax.legend(loc='upper left')
save(fig, "histogram_number_of_solvents.png")
print(f"Soluble nowhere: {(n_sol == 0).sum()} | everywhere: {(n_sol == 5).sum()}")

### 3. Overlaps between solvents

Venn diagrams with 3 solvents (beyond that, a Venn diagram becomes unreadable). Each area gives the number of molecules soluble in those solvents and not in the other ones of the diagram. The next chart gives, for all 5 solvents at once, the most frequent exact combinations.

In [ ]:
def venn(trio, name):
    fig, ax = plt.subplots(figsize=(5.6, 4.8))
    sets = [set(df.index[df[s] == 1]) for s in trio]
    v = venn3(sets, set_labels=[LABEL[s] for s in trio], set_colors=[ps.SOLVENT_COLORS[s] for s in trio], alpha=0.4, ax=ax)
    for t in v.set_labels:
        if t: t.set_fontsize(15); t.set_fontweight('bold')
    for t in v.subset_labels:
        if t: t.set_fontsize(13)
    save(fig, name)

venn(('DMSO', 'DCM', 'CHCl3'), "venn_dmso_dcm_chcl3.png")
venn(('DMSO', 'MeOH', 'ACN'), "venn_dmso_meoh_acn.png")

In [ ]:
combo = df[SOLVENTS].fillna(0).astype(int).apply(lambda r: " + ".join(s for s in SOLVENTS if r[s]) or "none", axis=1)
top = combo.value_counts().head(10)[::-1]
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(top.index.str.replace('CHCl3', 'CHCl$_3$'), top.values, color=ps.GREY, height=0.66)
for i, v in enumerate(top.values):
    ax.text(v + 0.8, i, str(v), va='center', fontsize=12)
ax.grid(axis='y', visible=False); ax.grid(axis='x', visible=True); ax.set_xlabel("molecules")
ax.tick_params(axis='y', length=0)
save(fig, "solvent_combinations.png")

### 4. Which solvents behave alike?

**Agreement**: share of molecules with the same result (soluble or insoluble) in both solvents. This is the measure to **group solvents that behave alike**.

**Conditional probability**: cell (row A, column B) gives the probability that a molecule is soluble in B **given** that it is soluble in A.

In [ ]:
L = df[SOLVENTS]
agree = pd.DataFrame({b: [(L[a] == L[b]).mean() for a in SOLVENTS] for b in SOLVENTS}, index=SOLVENTS)
cond = pd.DataFrame({b: [L.loc[L[a] == 1, b].mean() for a in SOLVENTS] for b in SOLVENTS}, index=SOLVENTS)

def heatmap(mat, name, vmin):
    fig, ax = plt.subplots(figsize=ps.SQUARE)
    ax.imshow(mat.values, cmap=REDS, vmin=vmin, vmax=1)
    ax.set_xticks(range(5), [LABEL[s] for s in SOLVENTS]); ax.set_yticks(range(5), [LABEL[s] for s in SOLVENTS])
    ax.grid(False); ax.tick_params(length=0); ax.spines[:].set_visible(False)
    ax.xaxis.tick_top()
    for i in range(5):
        for j in range(5):
            v = mat.values[i, j]
            ax.text(j, i, f"{v:.0%}", ha='center', va='center', fontsize=13,
                    color='white' if (v - vmin) / (1 - vmin) > 0.55 else ps.INK)
    save(fig, name)

heatmap(agree, "solvent_agreement.png", vmin=0.5)
heatmap(cond, "solvent_conditional_probability.png", vmin=0.5)

pairs = sorted(((agree.loc[a, b], a, b) for a, b in itertools.combinations(SOLVENTS, 2)), reverse=True)
print("Most similar solvent pairs (agreement):")
for v, a, b in pairs:
    print(f"  {a:5s} - {b:5s}: {v:.0%}")

### 5. Solubility vs RT and molecular weight

Molecules are sorted into 5 equal-size groups (quintiles) by RT, then by molecular weight. For each group, we compute the share of molecules soluble in each solvent.

In [ ]:
for var, label, name in [('RT', "RT (min), group median", "solubility_vs_rt.png"),
                         ('MolWt', "molecular weight (g/mol), group median", "solubility_vs_molwt.png")]:
    fig, ax = plt.subplots(figsize=ps.HALF)
    bins = pd.qcut(df[var], 5)
    centers = df.groupby(bins, observed=True)[var].median().values
    ends = []
    for s in SOLVENTS:
        p = df.groupby(bins, observed=True)[s].mean().values
        ax.plot(centers, p, '-o', color=ps.SOLVENT_COLORS[s], ms=7, mec='white', mew=1.5)
        ends.append(p[-1])
    ax.set_ylim(0.2, 1.0); ax.yaxis.set_major_formatter(ps.pct)
    ax.set_xlim(centers[0] - 0.05 * (centers[-1] - centers[0]), centers[-1] + 0.2 * (centers[-1] - centers[0]))
    ps.label_line_ends(ax, centers, ends, [LABEL[s] for s in SOLVENTS])
    ax.set_xlabel(label); ax.set_ylabel("soluble molecules")
    save(fig, name)

### 6. The training set: original data vs plates

Do the plates look like the original data? If their molecules have very different RTs or weights, the model learns in new regions. That is useful, but it must be known to interpret the scores.

In [ ]:
for var, label, name in [('RT', "RT (min)", "distribution_rt.png"), ('MolWt', "molecular weight (g/mol)", "distribution_molwt.png")]:
    fig, ax = plt.subplots(figsize=ps.HALF)
    bins = np.histogram_bin_edges(df[var].dropna(), bins=14)
    for src in ['original', 'plates']:
        d = df[df['source'] == src][var].dropna()
        if len(d):
            ax.hist(d, bins=bins, histtype='step', lw=2.5, color=SOURCE_COLORS[src], density=True,
                    label=f"{src} (median {d.median():.0f})" if var == 'MolWt' else f"{src} (median {d.median():.1f})")
    ax.set_xlabel(label); ax.set_yticks([]); ax.set_ylabel("share of molecules"); ax.grid(False)
    ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncols=2)
    save(fig, name)

summary = df.groupby('source').agg(molecules=('CAS', 'size'), RT_median=('RT', 'median'), MolWt_median=('MolWt', 'median'),
                                   not_BlueBird_1=('hplc_column', lambda c: (c != 'BlueBird 1').sum()))
display(summary)

## Part B: the model

This part evaluates **Model E** (Arthur's final model, the one in notebooks 01 to 04): one row per molecule x HPLC column x solvent, with RT, molecular weight, the solvent's dielectric constant and Hansen parameters, and the 10 standards of the HPLC column.

The overall score of Model E is in notebook `04_robustness_check`; this part goes further. All results come from the same kind of **cross-validation**: the molecules are split into 5 groups, each group is the test set once, and the whole procedure is repeated with different splits. Every molecule is therefore predicted by a model that has never seen it. A molecule measured on several HPLC columns gets the **mean** of the probabilities predicted on its columns.

This part reads `data/interim/model_e_dataset.xlsx`, built by `scripts/data_prep/add_new_molecules.py`. Figures are saved to `results/figures/model_analysis/`.

### Settings

In [ ]:
N_FOLDS = 5                                     # 5-fold cross-validation
N_REPEATS = 3                                   # number of different splits (more = more stable, but slower)
FRACTIONS = [0.2, 0.4, 0.6, 0.8, 1.0]           # training-set sizes for the learning curve

In [ ]:
from config import MODEL_E_DATASET, MATRIX_IDS_TABLE, FIGURES, ensure_dir
from data_converter.hplc_data_handler import extract_solvent_physics

import xgboost as xgb
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

FIG_MODEL = ensure_dir(FIGURES / "model_analysis")

def save_model_fig(fig, name):
    fig.savefig(FIG_MODEL / name)
    plt.show()

def make_model_E(weight, seed):
    """Arthur's Model E settings (rt_prediction_continued.ipynb, cell 18)."""
    return xgb.XGBClassifier(n_estimators=1000, learning_rate=0.01, max_depth=4, subsample=0.8, colsample_bytree=0.5,
                             colsample_bynode=0.9, gamma=.8, min_child_weight=10, reg_lambda=5, reg_alpha=1,
                             scale_pos_weight=weight, random_state=seed, n_jobs=-1)

def first_choice(df):
    """For each molecule: the solvent ranked first and whether that choice was right."""
    keys = [c for c in ['rep'] if c in df.columns] + ['CAS']
    rows = []
    for values, g in df.groupby(keys):
        values = values if isinstance(values, tuple) else (values,)
        best = g.loc[g['p'].idxmax()]
        dmso = g[g['Solvent'] == 'DMSO']
        rows.append(dict(zip(keys, values), choice=best['Solvent'], correct=best['y'] == 1,
                         soluble_somewhere=g['y'].max() == 1,
                         dmso_correct=bool(len(dmso)) and dmso['y'].iloc[0] == 1,
                         dmso_insoluble=bool(len(dmso)) and dmso['y'].iloc[0] == 0))
    return pd.DataFrame(rows)

### 1. Preparing the data

The same features as in notebook 01. Molecules are identified by their CAS.

In [ ]:
_molwt = {}
def molecular_weight(smiles):
    if smiles not in _molwt:
        mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
        _molwt[smiles] = Descriptors.MolWt(mol) if mol else np.nan
    return _molwt[smiles]

physics = {s: extract_solvent_physics(s).values for s in SOLVENTS}   # dielectric, Hansen D, P, H
standards = pd.read_excel(MATRIX_IDS_TABLE).set_index('MatrixID')[[f'M{i}' for i in range(1, 11)]]

# one row per molecule x HPLC column x solvent
e = pd.read_excel(MODEL_E_DATASET)
e['CAS'] = e['CAS '].astype(str).str.strip()
e = e[(e['RT'] != 0) & e['solute_smiles'].notna()]
e['Matrix'] = e['Matrix ID Name'].str.replace('matrix ID ', '', regex=False).str.strip()
e = e.melt(id_vars=['CAS', 'RT', 'Matrix', 'solute_smiles'], value_vars=SOLVENTS,
           var_name='Solvent', value_name='Soluble').dropna(subset=['Soluble'])
e['MolWt'] = e['solute_smiles'].map(molecular_weight)
e = e.dropna(subset=['MolWt']).reset_index(drop=True)
X = np.hstack([e[['RT', 'MolWt']].values,
               np.vstack(e['Solvent'].map(physics).values),
               np.vstack(e['Matrix'].map(lambda m: standards.loc[m].values.astype(float)).values)])
y = e['Soluble'].astype(int).values
molecules = np.array(sorted(e['CAS'].unique()))
print(f"{len(e)} rows, {len(molecules)} molecules")

### 2. Cross-validation

For each split and each training-set size, Model E is trained and then predicts the molecules of the test fold.

This cell takes a few minutes.

In [ ]:
def predict_fold(train_mask, test_mask, seed):
    """Train Model E on train_mask, return one row per test molecule x solvent (mean probability over its columns)."""
    model = make_model_E((y[train_mask] == 0).sum() / (y[train_mask] == 1).sum(), seed).fit(X[train_mask], y[train_mask])
    pred = e.loc[test_mask, ['CAS', 'Solvent']].assign(p=model.predict_proba(X[test_mask])[:, 1], y=y[test_mask])
    return pred.groupby(['CAS', 'Solvent'], as_index=False).agg(p=('p', 'mean'), y=('y', 'first'))

predictions = []
for rep in range(N_REPEATS):
    order = np.random.RandomState(rep).permutation(len(molecules))
    fold = pd.Series(np.arange(len(molecules)) % N_FOLDS, index=molecules[order])
    for k in range(N_FOLDS):
        test_cas = set(fold[fold == k].index)
        train_cas = np.array(sorted(set(molecules) - test_cas))
        for frac in FRACTIONS:
            sub_train = set(np.random.RandomState(100 * rep + k).permutation(train_cas)[:int(round(frac * len(train_cas)))])
            pred = predict_fold(e['CAS'].isin(sub_train).values, e['CAS'].isin(test_cas).values, rep)
            predictions.append(pred.assign(rep=rep, fold=k, fraction=frac, n_train=len(sub_train)))
    print(f"split {rep + 1}/{N_REPEATS} done")

res = pd.concat(predictions, ignore_index=True)
res['pred'] = (res['p'] >= 0.5).astype(int)
full = res[res['fraction'] == 1.0].copy()      # models trained on the whole training set
top1 = first_choice(full)                      # solvent ranked first for each molecule
print(f"Correct solvent choice (top-1): {top1['correct'].mean():.1%}   (always DMSO: {top1['dmso_correct'].mean():.1%})")

### 3. Scores per solvent

Accuracy of Model E for each solvent, compared with the best constant answer for that solvent (grey line). A bar below the grey line means that, for this solvent, the model does no better than giving the same answer everywhere.

In [ ]:
per_solvent = full.groupby('Solvent').apply(lambda g: accuracy_score(g['y'], g['pred']), include_groups=False).reindex(SOLVENTS)
ref_solvent = full.groupby('Solvent')['y'].mean().reindex(SOLVENTS)
ref_solvent = np.maximum(ref_solvent, 1 - ref_solvent)    # best constant answer (always soluble or always insoluble)

fig, ax = plt.subplots(figsize=ps.HALF)
x = np.arange(len(SOLVENTS))
ax.set_ylim(0, 1.05)
bars = ax.bar(x, per_solvent.values, 0.6, color=ps.RED, label='Model E')
ps.label_bars(ax, bars, inside=True, color='white')
for j, v in enumerate(ref_solvent.values):
    ax.hlines(v, j - 0.38, j + 0.38, color=ps.DARK, lw=2.5, linestyles=(0, (2, 1.5)), label='constant answer' if j == 0 else None)
ax.set_xticks(x, [LABEL[s] for s in SOLVENTS]); ax.yaxis.set_major_formatter(ps.pct); ax.set_yticks([0, .25, .5, .75, 1])
ax.set_ylabel("accuracy"); ax.legend(ncols=2, loc='upper left', bbox_to_anchor=(0, 1.1))
save_model_fig(fig, "accuracy_per_solvent.png")
display(pd.DataFrame({'Model E': per_solvent, 'constant answer': ref_solvent}).style.format("{:.0%}"))

#### Confusion matrices

For each solvent: rows are the truth, columns the prediction (mean number of molecules per split). The errors to watch are **"insoluble" predicted "soluble"** (top right): the model sends the molecule to a solvent where it will not dissolve.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(13, 3.4), gridspec_kw={"wspace": 0.12})
for ax, s in zip(axes, SOLVENTS):
    gs = full[full['Solvent'] == s]
    cm = confusion_matrix(gs['y'], gs['pred'], labels=[0, 1]) / N_REPEATS
    ax.imshow(cm, cmap=REDS, vmin=0, vmax=cm.max() * 1.15); ax.grid(False); ax.spines[:].set_visible(False)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:.0f}", ha='center', va='center', fontsize=15,
                    color='white' if cm[i, j] > cm.max() * 0.6 else ps.INK)
    ax.set_xticks([0, 1], ['insol.', 'sol.']); ax.set_yticks([0, 1], ['insol.', 'sol.'] if s == SOLVENTS[0] else [])
    ax.tick_params(length=0); ax.set_title(LABEL[s], fontsize=14, fontweight='bold'); ax.set_xlabel('predicted')
axes[0].set_ylabel('actual')
save_model_fig(fig, "confusion_matrices_model_e.png")

### 4. Learning curve

The model is retrained on only part of the training molecules (20 %, 40 % ... 100 %), then tested on the same test molecules. If the curve **is still rising** on the right, adding molecules should improve the model. If it is **flat**, more data of the same kind will not help much.

In [ ]:
def scores(g):
    return pd.Series({'top1': first_choice(g)['correct'].mean(), 'bacc': balanced_accuracy_score(g['y'], g['pred'])})

curve = res.groupby(['fraction', 'rep']).apply(scores, include_groups=False).reset_index()
n_train = res.groupby('fraction')['n_train'].mean()
curve_summary = curve.drop(columns='rep').groupby('fraction').agg(['mean', 'std'])
ref_dmso = top1['dmso_correct'].mean()

for col, ylabel, ref, ref_name, name in [('top1', "correct first solvent (top-1)", ref_dmso, "always DMSO", "learning_curve_top1.png"),
                                         ('bacc', "balanced accuracy", 0.5, "random guess", "learning_curve_balanced_accuracy.png")]:
    fig, ax = plt.subplots(figsize=ps.HALF)
    mean, std = curve_summary[(col, 'mean')], curve_summary[(col, 'std')]
    xs = n_train.loc[mean.index].values
    ax.fill_between(xs, (mean - std).values, (mean + std).values, color=ps.RED, alpha=0.12, lw=0)
    ax.plot(xs, mean.values, '-o', color=ps.RED, mec='white', mew=1.5)
    ax.hlines(ref, xs[0], xs[-1], color=ps.DARK, ls=(0, (2, 1.5)), lw=2)
    ax.text(xs[-1] + 4, ref, ref_name, va='center', fontsize=12, color=ps.DARK)
    ax.text(xs[-1], mean.values[-1], "  Model E", va='center', fontsize=12)
    ax.set_xlim(xs[0] - 5, xs[-1] + 40); ax.yaxis.set_major_formatter(ps.pct)
    ax.set_xlabel("training molecules"); ax.set_ylabel(ylabel)
    save_model_fig(fig, name)
display(curve_summary.style.format("{:.1%}"))

### 5. Probability calibration

Predictions are grouped into bins of announced probability (0-20 %, 20-40 % ...). For each bin, the mean announced probability is compared with the share of molecules **actually** soluble. If the points follow the diagonal, the model is well calibrated: when it announces 80 %, the molecule dissolves 8 times out of 10. This is essential for a future cost function, which will rely on these probabilities.

The **Brier score** summarises calibration: 0 is perfect, 0.25 corresponds to always announcing 50 %.

In [ ]:
full['bin'] = pd.cut(full['p'], np.linspace(0, 1, 6), include_lowest=True)
calib = full.groupby('bin', observed=True).agg(announced_proba=('p', 'mean'), actual_share=('y', 'mean'), predictions=('y', 'size'))
display(calib.style.format({'announced_proba': '{:.0%}', 'actual_share': '{:.0%}', 'predictions': '{:.0f}'}))
brier = np.mean((full['p'] - full['y']) ** 2)
brier_ref = np.mean((full['y'].mean() - full['y']) ** 2)
print(f"Brier score: {brier:.3f}   (reference, always announcing the mean share of soluble: {brier_ref:.3f})")

fig, ax = plt.subplots(figsize=ps.SQUARE)
ax.plot([0, 1], [0, 1], color=ps.DARK, ls=(0, (2, 1.5)), lw=2)
ax.text(0.22, 0.30, "perfect calibration", rotation=45, rotation_mode='anchor', fontsize=12, color=ps.DARK)
ax.plot(calib['announced_proba'], calib['actual_share'], '-o', color=ps.RED, ms=10, mec='white', mew=1.5)
for _, r in calib.iterrows():
    ax.annotate(f"n={r['predictions']:.0f}", (r['announced_proba'], r['actual_share']), xytext=(0, 14),
                textcoords='offset points', fontsize=11, color=ps.MUTED, ha='center')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.xaxis.set_major_formatter(ps.pct); ax.yaxis.set_major_formatter(ps.pct)
ax.grid(axis='x', visible=True)
ax.set_xlabel("announced probability of being soluble"); ax.set_ylabel("actually soluble")
save_model_fig(fig, "calibration_model_e.png")

### 6. Top-1, top-2, top-3: how many attempts are needed?

For each molecule soluble in at least one solvent, we follow the model's ranking and look at which attempt hits a solvent that works. "Top-2" = a working solvent is among the first 2 suggested.

Reference: **one fixed order** for all molecules, from the most often to the least often soluble solvent.

Top-1 is a bit higher here than the top-1 of notebook 04, because molecules soluble **nowhere** are excluded: for them, no order can find a working solvent.

In [ ]:
fixed_order = list(full.groupby('Solvent')['y'].mean().sort_values(ascending=False).index)
print("Fixed reference order:", " > ".join(fixed_order))

def rank_of_first_hit(g, order=None):
    """Position (1 = first attempt) of the first solvent that works."""
    ranking = g.sort_values('p', ascending=False)['Solvent'].tolist() if order is None else [s for s in order if s in set(g['Solvent'])]
    soluble = set(g.loc[g['y'] == 1, 'Solvent'])
    return next(i + 1 for i, s in enumerate(ranking) if s in soluble)

ranks = []
for (rep, cas), g in full.groupby(['rep', 'CAS']):
    if g['y'].max() == 1:
        ranks.append({'method': 'Model E', 'rank': rank_of_first_hit(g)})
        ranks.append({'method': 'Fixed order', 'rank': rank_of_first_hit(g, fixed_order)})
ranks = pd.DataFrame(ranks)
topk = pd.DataFrame({k: ranks.groupby('method')['rank'].apply(lambda r: (r <= k).mean()) for k in range(1, 6)}).T
topk.index = [f"top-{k}" for k in topk.index]
display(topk[['Model E', 'Fixed order']].style.format("{:.1%}"))
print("Mean number of attempts:", ranks.groupby('method')['rank'].mean().round(2).to_dict())

fig, ax = plt.subplots(figsize=ps.HALF)
ks = np.arange(1, 6)
ax.plot(ks, topk['Fixed order'].values, '--o', color=ps.GREY, mec='white', mew=1.5)
ax.plot(ks, topk['Model E'].values, '-o', color=ps.RED, mec='white', mew=1.5)
ax.text(0.85, topk['Model E'].iloc[0], "Model E", va='center', ha='right', fontsize=12)
ax.text(0.85, topk['Fixed order'].iloc[0], "fixed order", va='center', ha='right', fontsize=12, color=ps.DARK)
ax.text(5, min(topk.min()) - 0.01, "fixed order: " + " > ".join(LABEL[s] for s in fixed_order),
        ha='right', va='top', fontsize=11, color=ps.MUTED)
ax.set_xlim(-0.2, 5.25)
ax.set_xticks(ks, [f"top-{k}" for k in ks]); ax.yaxis.set_major_formatter(ps.pct)
ax.set_ylim(min(topk.min()) - 0.05, 1.01); ax.set_ylabel("molecules with a working solvent")
save_model_fig(fig, "top_k.png")

### 7. Feature importance

To measure a feature's importance, it is **randomly shuffled** in the test data (the model can no longer use it) and we look at how much the balanced accuracy drops. The bigger the drop, the more the model relies on that feature. The 4 solvent properties and the 10 column standards are each shuffled together, as one block.

In [ ]:
feature_groups = {'RT': [0], 'Molecular weight': [1], 'Solvent properties': [2, 3, 4, 5], 'HPLC column standards': list(range(6, 16))}
rng = np.random.RandomState(0)
mol_order = np.random.RandomState(0).permutation(len(molecules))
cv_fold = pd.Series(np.arange(len(molecules)) % N_FOLDS, index=molecules[mol_order])
drops = []
for k in range(N_FOLDS):
    te = e['CAS'].isin(set(cv_fold[cv_fold == k].index)).values
    model = make_model_E((y[~te] == 0).sum() / (y[~te] == 1).sum(), 0).fit(X[~te], y[~te])
    base = balanced_accuracy_score(y[te], model.predict(X[te]))
    for name, cols in feature_groups.items():
        for _ in range(5):
            Xp = X[te].copy()
            Xp[:, cols] = Xp[rng.permutation(len(Xp))][:, cols]
            drops.append({'feature': name, 'drop': base - balanced_accuracy_score(y[te], model.predict(Xp))})
imp = pd.DataFrame(drops).groupby('feature')['drop'].agg(['mean', 'std']).sort_values('mean')
display(imp.style.format("{:.3f}"))

fig, ax = plt.subplots(figsize=(6.6, 3.6))
ax.barh(imp.index, imp['mean'] * 100, color=ps.RED, height=0.58)
for i, v in enumerate(imp['mean'] * 100):
    ax.text(max(v, 0) + 0.15, i, f"{v:.1f} pts", va='center', fontsize=12)
ax.grid(axis='y', visible=False); ax.grid(axis='x', visible=True); ax.tick_params(axis='y', length=0)
ax.set_xlim(0, imp['mean'].max() * 100 * 1.25)
ax.set_xlabel("drop in balanced accuracy when shuffled (points)")
save_model_fig(fig, "feature_importance.png")

### 8. Error analysis

The molecule x solvent pairs that Model E gets **wrong in every split**. They are either genuinely atypical molecules or **doubtful labels** worth re-checking, especially those coming from the vision model.

In [ ]:
# origin of each label: original data, or plate (your observation / vision model)
wells = []
for path in experiment_files():
    recs, _ = read_experiment(path)
    wells += [dict(r, plate=path.stem.replace('_output', '')) for r in recs]
wells = pd.DataFrame(wells)
plate_of = wells.groupby('cas')['plate'].first() if len(wells) else pd.Series(dtype=str)
label_origin = {(r['cas'], r['solvent']): ('own observation' if r['from_observation'] else 'vision model') for _, r in wells.iterrows()}

names = pd.read_excel(MODEL_E_DATASET)
names = names.assign(CAS=names['CAS '].astype(str).str.strip()).drop_duplicates('CAS').set_index('CAS')['Sample Name']
info = e.groupby('CAS').agg(RT_median=('RT', 'median'), MolWt=('MolWt', 'first'))

err = full.assign(wrong=full['pred'] != full['y']).groupby(['CAS', 'Solvent']).agg(error_rate=('wrong', 'mean'), actual=('y', 'first'), proba=('p', 'mean')).reset_index()
err['Molecule'] = err['CAS'].map(names)
err['Source'] = err['CAS'].map(plate_of).fillna("original data")
err['Label'] = [label_origin.get((c, s), "original data") for c, s in zip(err['CAS'], err['Solvent'])]
err = err.join(info, on='CAS')

by_label = err.groupby('Label')['error_rate'].agg(['mean', 'size']).reindex(['original data', 'vision model', 'own observation']).dropna()
display(by_label.rename(columns={'mean': 'error rate', 'size': 'molecule x solvent pairs'}).style.format({'error rate': '{:.0%}'}))

fig, ax = plt.subplots(figsize=ps.HALF)
ax.set_ylim(0, 0.8)
bars = ax.bar(range(len(by_label)), by_label['mean'], 0.58,
              color=[ps.GREY if l == 'original data' else ps.RED for l in by_label.index])
ps.label_bars(ax, bars)
ax.set_xticks(range(len(by_label)), [f"{l}\n({n:.0f} pairs)" for l, n in zip(by_label.index, by_label['size'])])
ax.yaxis.set_major_formatter(ps.pct); ax.set_yticks([0, .2, .4, .6, .8]); ax.set_ylabel("error rate (Model E)")
save_model_fig(fig, "error_rate_by_label_origin.png")

always_wrong = err[err['error_rate'] == 1].assign(gap=lambda x: (x['proba'] - x['actual']).abs()).sort_values('gap', ascending=False)
print(f"{len(always_wrong)} molecule x solvent pairs always wrong (out of {len(err)}). The 20 clearest:")
display(always_wrong.head(20)[['Molecule', 'CAS', 'Solvent', 'actual', 'proba', 'RT_median', 'MolWt', 'Source', 'Label']]
        .rename(columns={'actual': 'actual (1 = soluble)', 'proba': 'predicted proba', 'RT_median': 'RT (median)'})
        .style.format({'predicted proba': '{:.0%}', 'RT (median)': '{:.2f}', 'MolWt': '{:.0f}'}).hide(axis='index'))

### 9. Effect of each plate

Model E is retrained adding the plates **one by one**, in the order of their names. The tested molecules always stay the same (all of them, plates included), with the same splits: only the training set changes. We track the correct solvent choice over all molecules, and over those **insoluble in DMSO**, the case the plates target.

This cell takes one to two minutes.

In [ ]:
plate_list = sorted(plate_of.unique())
variants = [("original data", set())]
for i in range(len(plate_list)):
    variants.append(("+ " + plate_list[i].replace('MatrixID_', ''), set(plate_list[:i + 1])))

effect = []
for name, included in variants:
    excluded = set(plate_of[~plate_of.isin(included)].index)       # molecules of plates not added yet
    for rep in range(N_REPEATS):
        mol_order = np.random.RandomState(rep).permutation(len(molecules))
        cv_fold = pd.Series(np.arange(len(molecules)) % N_FOLDS, index=molecules[mol_order])
        preds = []
        for k in range(N_FOLDS):
            te = e['CAS'].isin(set(cv_fold[cv_fold == k].index)).values
            tr = ~te & ~e['CAS'].isin(excluded).values
            preds.append(predict_fold(tr, te, rep))
        pr = pd.concat(preds)
        t1 = first_choice(pr)
        effect.append({'variant': name, 'rep': rep, 'molecules added': len(plate_of) - len(excluded),
                       'correct choice (all)': t1['correct'].mean(),
                       'correct choice (DMSO-insoluble)': t1.loc[t1['dmso_insoluble'], 'correct'].mean(),
                       'balanced accuracy': balanced_accuracy_score(pr['y'], (pr['p'] >= 0.5).astype(int))})
    print(f"{name}: done")

effect = pd.DataFrame(effect).groupby('variant', sort=False).mean(numeric_only=True).drop(columns='rep')
display(effect.style.format({'molecules added': '{:.0f}', 'correct choice (all)': '{:.1%}',
                             'correct choice (DMSO-insoluble)': '{:.1%}', 'balanced accuracy': '{:.1%}'}))

fig, ax = plt.subplots(figsize=(7.6, 4.4))
xs = np.arange(len(effect))
for col, color, label in [('correct choice (all)', ps.DARK, "all molecules"),
                          ('correct choice (DMSO-insoluble)', ps.RED, "DMSO-insoluble molecules")]:
    vals = effect[col].values
    ax.plot(xs, vals, '-o', color=color, mec='white', mew=1.5)
    ax.text(xs[-1] + 0.12, vals[-1], f"{vals[-1]:.0%}  {label}", va='center', fontsize=12)
    ax.text(xs[0] - 0.12, vals[0], f"{vals[0]:.0%}", va='center', ha='right', fontsize=12)
ax.set_xticks(xs, effect.index); ax.set_xlim(-0.5, len(xs) + 1.3)
ax.yaxis.set_major_formatter(ps.pct); ax.set_ylim(0, 1); ax.set_ylabel("correct first solvent (top-1)")
save_model_fig(fig, "plate_effect.png")